In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_timestamp, count, sum as _sum, avg, round as _round,
    min as _min, max as _max, window
)

spark = (
    SparkSession.builder
    .appName("PracaDomowa-Spark")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [2]:
df = spark.read.json("transactions_10k.jsonl")

df.show(5)
print(df.count())

+------+-----------+--------+-------------------+-------+-------+
|amount|   category|   store|          timestamp|  tx_id|user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|    u48|
| 79.57|    książki|Warszawa|2026-04-12 08:05:43|TX00002|    u15|
|126.17|     odzież|Warszawa|2026-04-12 09:15:30|TX00003|    u18|
| 34.08|     odzież|Warszawa|2026-04-12 10:05:39|TX00004|    u10|
|428.88|    żywność|  Kraków|2026-04-12 09:04:36|TX00005|    u17|
+------+-----------+--------+-------------------+-------+-------+
only showing top 5 rows

10000


In [3]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss")
)

df.printSchema()

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [5]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)

store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [6]:
from pyspark.sql.functions import window, avg

gdansk_hourly_avg = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(avg("amount"), 2).alias("srednia_PLN")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "srednia_PLN"
    )
    .orderBy("srednia_PLN")
)

gdansk_hourly_avg.show(truncate=False)

+-------------------+-------------------+-----------+
|od                 |do                 |srednia_PLN|
+-------------------+-------------------+-----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|395.01     |
|2026-04-12 10:00:00|2026-04-12 11:00:00|412.92     |
|2026-04-12 09:00:00|2026-04-12 10:00:00|415.91     |
+-------------------+-------------------+-----------+



In [7]:
from pyspark.sql.functions import hour, minute, count

category_0900_0930 = (
    df.filter(
        (hour(col("timestamp")) == 9) &
        (minute(col("timestamp")) < 30)
    )
    .groupBy("category")
    .agg(count("tx_id").alias("liczba_tx"))
    .orderBy("category")
)

category_0900_0930.show(truncate=False)

+-----------+---------+
|category   |liczba_tx|
+-----------+---------+
|elektronika|611      |
|książki    |622      |
|odzież     |605      |
|żywność    |567      |
+-----------+---------+



In [8]:
from pyspark.sql.functions import to_timestamp, lit

category_0900_0930 = (
    df.filter(
        (col("timestamp") >= to_timestamp(lit("2026-04-12 09:00:00"))) &
        (col("timestamp") <  to_timestamp(lit("2026-04-12 09:30:00")))
    )
    .groupBy("category")
    .agg(count("tx_id").alias("liczba_tx"))
    .orderBy("category")
)

category_0900_0930.show()

+-----------+---------+
|   category|liczba_tx|
+-----------+---------+
|elektronika|      611|
|    książki|      622|
|     odzież|      605|
|    żywność|      567|
+-----------+---------+



In [9]:
quarter_peak = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("liczba_tx"))
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx"
    )
    .orderBy(col("liczba_tx").desc())
)

quarter_peak.show(truncate=False)

+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
|2026-04-12 09:00:00|2026-04-12 09:15:00|1171     |
|2026-04-12 09:30:00|2026-04-12 09:45:00|1156     |
|2026-04-12 08:45:00|2026-04-12 09:00:00|1139     |
|2026-04-12 09:45:00|2026-04-12 10:00:00|1100     |
|2026-04-12 08:30:00|2026-04-12 08:45:00|899      |
|2026-04-12 10:00:00|2026-04-12 10:15:00|858      |
|2026-04-12 08:15:00|2026-04-12 08:30:00|644      |
|2026-04-12 10:15:00|2026-04-12 10:30:00|582      |
|2026-04-12 08:00:00|2026-04-12 08:15:00|468      |
|2026-04-12 10:30:00|2026-04-12 10:45:00|443      |
|2026-04-12 10:45:00|2026-04-12 11:00:00|306      |
+-------------------+-------------------+---------+



In [12]:
print("1. Gdańsk – godzina z najniższą średnią kwotą:")
gdansk_hourly_avg.show(truncate=False)

print("2. Liczba transakcji per kategoria w oknie 09:00–09:30:")
category_0900_0930.show(truncate=False)

print("3. Ćwierćgodzina ze szczytem transakcji:")
quarter_peak.show(truncate=False)

1. Gdańsk – godzina z najniższą średnią kwotą:
+-------------------+-------------------+-----------+
|od                 |do                 |srednia_PLN|
+-------------------+-------------------+-----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|395.01     |
|2026-04-12 10:00:00|2026-04-12 11:00:00|412.92     |
|2026-04-12 09:00:00|2026-04-12 10:00:00|415.91     |
+-------------------+-------------------+-----------+

2. Liczba transakcji per kategoria w oknie 09:00–09:30:
+-----------+---------+
|category   |liczba_tx|
+-----------+---------+
|elektronika|611      |
|książki    |622      |
|odzież     |605      |
|żywność    |567      |
+-----------+---------+

3. Ćwierćgodzina ze szczytem transakcji:
+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
|2026-04-12 09:00:00|2026-04-12 09:15:00|1171     |
|2026-04-12 09:30:00|202